In [1]:
import numpy as np
import warnings

class ART2A:
    def __init__(self, M, theta, alpha, rho_star, beta):
        """
        Initialize the ART 2-A network exactly as specified by Carpenter, Grossberg, and Rosen (1991).
        
        Parameters:
        M : int
            Dimensionality of the input vectors.
        theta : float
            Signal threshold. Must satisfy 0 < theta <= 1/sqrt(M) as per Eq (61)[cite: 428].
        alpha : float
            Uncommitted node choice parameter. Must satisfy alpha <= 1/sqrt(M) as per Eq (63)[cite: 450].
        rho_star : float
            Vigilance parameter. Must satisfy 0 <= rho_star <= 1 as per Eq (66)[cite: 465].
        beta : float
            Learning rate for intermediate learning (beta=1 for fast learning). 
            Must satisfy 0 <= beta <= 1 as per Eq (70)[cite: 490].
        """
        # Calculate the upper bound for theta and alpha based on Eq (61) and Eq (63) [cite: 428, 450]
        limit = 1.0 / np.sqrt(M)
        
        # Validate theta against Eq (61): 0 < theta <= 1/sqrt(M) [cite: 428]
        if not (0 < theta <= limit):
            warnings.warn(f"theta={theta} violates constraint 0 < theta <= 1/sqrt(M) ({limit:.4f})")
            
        # Validate alpha against Eq (63): alpha <= 1/sqrt(M) [cite: 450]
        if alpha > limit:
            warnings.warn(f"alpha={alpha} violates constraint alpha <= 1/sqrt(M) ({limit:.4f})")
            
        # Assign hyperparameters to the instance
        self.theta = theta
        self.alpha = alpha
        self.rho_star = rho_star
        self.beta = beta
        
        # Initialize an empty list to store the scaled Long Term Memory (LTM) weight vectors z_j^* # for committed nodes. Initially, all F2 nodes are uncommitted[cite: 452].
        self.W = [] 

    def _normalize(self, x):
        """Operator N: Carries out Euclidean normalization based on Eq (59)[cite: 419]."""
        # Calculate the Euclidean norm (||x||) of the vector
        norm = np.linalg.norm(x)
        # Return x / ||x|| if norm is non-zero, otherwise return a zero vector to avoid division by zero [cite: 419]
        return x / norm if norm > 0 else np.zeros_like(x)

    def _threshold(self, x):
        """Operator F_theta: Sets subthreshold signals to zero based on Eq (60)[cite: 427]."""
        # (F_theta x)_i = x_i if x_i > theta, otherwise 0 [cite: 427]
        return np.where(x > self.theta, x, 0.0)

    def process_input(self, I0):
        """
        Processes a single M-dimensional input vector I^0 and updates the network.
        Returns the index of the activated F2 category node.
        """
        # Ensure the raw input I^0 is a numpy array of floats for mathematical operations 
        I0 = np.asarray(I0, dtype=float)
        
        # --- 1. Input Processing ---
        # The F0 -> F1 input vector I satisfies Eq (58): I = N * F_theta * N * I^0 
        
        # Step 1: Normalize the raw input I^0 -> N * I^0
        I_norm = self._normalize(I0)
        
        # Step 2: Apply the threshold function -> F_theta * (N * I^0)
        I_thresh = self._threshold(I_norm)
        
        # Step 3: Normalize the thresholded result -> N * (F_theta * N * I^0) to get final input I
        I = self._normalize(I_thresh)
        
        # --- 2. F2 Activation (Choice Function) ---
        # Calculate F1 -> F2 input signals T_j based on Eq (62) 
        
        # For committed nodes: T_j = I \cdot z_j^* (since ||I||=1 and ||z_j^*||=1, this is a cosine similiriaty) 
        T_committed = [np.dot(I, z_star) for z_star in self.W] 
        
        # For uncommitted nodes: T_j = alpha * sum(I_i) [cite: 447]
        T_uncommitted = self.alpha * np.sum(I) 
        
        # Determine initial choice J by finding the node that maximizes T_J, per Eq (64): T_J = max(T_j) 
        if len(self.W) > 0:
            # Find the index of the committed node with the highest activation
            J_committed = np.argmax(T_committed)
            # Store the highest activation value among committed nodes
            max_T_committed = T_committed[J_committed]
        else:
            # If there are no committed nodes yet, default the max committed activation to -1.0
            max_T_committed = -1.0
            
        # --- 3. Resonance or Reset ---
        # Flag to track whether we settle on an uncommitted node or a committed node
        is_uncommitted_chosen = False
        
        # Compare the highest committed node activation to the uncommitted node activation
        if max_T_committed >= T_uncommitted:
            # A committed node is the initial choice J 
            
            # Check vigilance criterion per Eq (65): T_J >= rho_star 
            if max_T_committed >= self.rho_star:
                # Resonance occurs: The node J remains constant 
                J = J_committed
            else:
                # Reset occurs per Eq (67): T_J < rho_star 
                # J is reset to the index of an arbitrary uncommitted node 
                is_uncommitted_chosen = True
        else:
            # An uncommitted node is initially chosen because it has the maximal T_j 
            is_uncommitted_chosen = True
            
        # --- 4. Learning ---
        # Update the scaled LTM vector z_J^* at the end of the input presentation according to Eq (68) 
        
        if is_uncommitted_chosen:
            # If J is an uncommitted node, set z_J^* equal to the current input I (Fast commitment) 
            self.W.append(I.copy())
            # The index J of this newly committed node is the last index in the W list
            J = len(self.W) - 1
        else:
            # If J is a committed node, perform slow/fast recoding 
            # Retrieve the LTM vector at the start of the input presentation: z_J^{*(old)} 
            z_old = self.W[J]
            
            # Define the critical feature vector Psi based on Eq (69) 
            # Psi_i = I_i if z_Ji^{*(old)} > theta, else 0 
            Psi = np.where(z_old > self.theta, I, 0.0)
            
            # LEARNING $
            # Calculate the new LTM vector based on Eq (68): z_J^{*(new)} = N(beta * Psi + (1 - beta) * z_J^{*(old)}) 
            z_new = self._normalize(self.beta * Psi + (1.0 - self.beta) * z_old)
            
            # Update the stored LTM vector for category J
            self.W[J] = z_new
            
        # Return the index of the category node that successfully coded the input I
        return J

In [4]:
import pandas as pd

# 1. Load the dataset from the same directory
df = pd.read_csv('data.csv')

# 2. Define the exact columns you specified for clustering
clustering_features = [
    'anonINDX', 
    'eINDX', 
    'HSCH', 
    'MJR', 
    'atitMATH', 
    'firstQIZZ', 
    'diffQIZZ', 
    'averQIZZ', 
    'numE', 
    'honLST', 
    'numP', 
    'numQ', 
    'averATND', 
    'numHW', 
    'honHW', 
    'paperHW', 
    'lateHW',
    'hardHW',
    'additHW',
    'lateADHW',
    'proxi.25',
    'proxi.75',
    'Y',
    'group',
    'CHEAT'
]

# 3. Create a new DataFrame with just the numerical clustering columns
clustering_df = df[clustering_features].copy()

# 4. Handle missing values 
# (ART 2-A's Euclidean math will break if it encounters a NaN. Filling with 0 is safest here)
clustering_df = clustering_df.fillna(0)

# 5. Convert everything to floats to strictly match the model's expected input type
clustering_df = clustering_df.astype(float)


In [21]:
import numpy as np

# 1. Define the network parameters
M = clustering_df.shape[1]      # M = 17 features
limit = 1.0 / np.sqrt(M)        # Upper bound constraint (~0.2425)

# Setting parameters based on the paper's fast-learn examples
theta = limit                   # Signal threshold
alpha = limit                   # Uncommitted node choice parameter
rho_star = 0.95            # Vigilance parameter (adjust to control cluster granularity)
beta = .5                      # Learning rate (1 = fast learning, 0.1 = slow learning)

# 2. Instantiate the model
art = ART2A(M=M, theta=theta, alpha=alpha, rho_star=rho_star, beta=beta)

# 3. Extract the first 50,000 observations
# .values converts the DataFrame to a numpy array for much faster iteration
data_subset = clustering_df.values[:50000]

# 4. Run the data through the network
print(f"Starting ART 2-A clustering on {len(data_subset)} observations...")

for i, row in enumerate(data_subset):
    art.process_input(row)
    
    # Optional: Print progress every 10,000 rows
    if (i + 1) % 10000 == 0:
        print(f"Processed {i + 1} rows... Current commited nodes: {len(art.W)}")

# 5. Print the final number of clusters (or nodes)
num_clusters = len(art.W)
print(f"\n Total number of commited nodes: {num_clusters}")

Starting ART 2-A clustering on 54 observations...

 Total number of commited nodes: 7


In [22]:
import numpy as np

# --- A. Assign each observation to the correct node/cluster ---

def predict_cluster(art_model, I0):
    """
    Finds the best matching F2 category without updating the LTM weights.
    Matches the choice function of the training phase.
    """
    I0 = np.asarray(I0, dtype=float)
    
    # 1. Input Processing
    I_norm = art_model._normalize(I0)
    I_thresh = art_model._threshold(I_norm)
    I = art_model._normalize(I_thresh)
    
    # 2. F2 Activation (Choice Function)
    if len(art_model.W) == 0:
        return -1 # Fallback in case no clusters exist
        
    # Calculate activations against committed nodes
    T_committed = [np.dot(I, z_star) for z_star in art_model.W]
    
    # Return the index of the highest activation
    return np.argmax(T_committed)

print("Assigning observations to clusters...")
# Generate the label array for the observations
cluster_labels = [predict_cluster(art, row) for row in data_subset]

# --- B. Print observations per cluster and the real centroids ---

print("\n--- Real-Space Cluster Summary ---")

# Count how many observations ended up in each cluster
unique_clusters, counts = np.unique(cluster_labels, return_counts=True)

# Convert cluster_labels to a numpy array for easy boolean indexing
labels_array = np.array(cluster_labels)

for cluster_idx, count in zip(unique_clusters, counts):
    if cluster_idx == -1:
        print("Warning: Unassigned observations found.")
        continue
        
    print(f"Cluster {cluster_idx}: {count} observations")
    
    # Isolate the original observations that belong to this cluster
    cluster_points = data_subset[labels_array == cluster_idx]
    
    # Calculate the actual real-space centroid (mean across the columns)
    real_centroid = np.mean(cluster_points, axis=0)
    
    # Round to 4 decimal places to keep it readable
    formatted_centroid = np.round(real_centroid, 4)
    
    print(f"Real-Space Centroid:\n{formatted_centroid}\n")
    print("-" * 60)



Assigning observations to clusters...

--- Real-Space Cluster Summary ---
Cluster 0: 1 observations
Real-Space Centroid:
[1.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
 0.   0.   0.   0.   0.   0.   0.   0.   0.15 1.   0.  ]

------------------------------------------------------------
Cluster 1: 10 observations
Real-Space Centroid:
[9.000e+00 3.500e+01 2.000e-01 1.000e+00 8.000e-01 1.495e-01 3.390e-02
 3.299e-01 3.000e+00 1.000e+00 2.900e+00 2.500e+00 2.800e+00 1.800e+00
 8.000e-01 3.667e-01 1.333e-01 4.056e-01 5.000e-01 2.000e-01 7.840e-01
 3.750e-01 2.925e-01 1.300e+00 6.000e-01]

------------------------------------------------------------
Cluster 2: 1 observations
Real-Space Centroid:
[ 5.     13.      1.      3.      3.      0.0625  0.1031  0.1656  4.
  1.      4.      3.      3.6667  1.      1.      0.3333  0.      0.2222
  0.      0.      1.      0.5     0.525   1.      1.    ]

------------------------------------------------------------
Cluster 3: 9 obse

In [23]:
import numpy as np

print("\n--- Cluster Profiles ---")

# 1. Get the global baseline to compare against
global_mean = np.mean(data_subset, axis=0)
global_std = np.std(data_subset, axis=0)

# Prevent division by zero just in case a feature has no variance
global_std[global_std == 0] = 1e-6 

features = np.array(clustering_features)
labels_array = np.array(cluster_labels)
unique_clusters, counts = np.unique(cluster_labels, return_counts=True)

for cluster_idx, count in zip(unique_clusters, counts):
    if cluster_idx == -1:
        continue
        
    cluster_points = data_subset[labels_array == cluster_idx]
    real_centroid = np.mean(cluster_points, axis=0)
    
    # 2. Calculate how far this centroid deviates from the global average (Z-scores)
    z_scores = (real_centroid - global_mean) / global_std
    
    # 3. Sort features by absolute z-score to find the most extreme/defining traits
    # argsort sorts ascending, so reverse it with [::-1]
    top_indices = np.argsort(np.abs(z_scores))[::-1]
    
    print(f"\nCLUSTER {cluster_idx} ({count} students):")
    print("Top 5 defining characteristics vs. the average student:")
    
    # Print the top 5 most distinguishing features
    for i in top_indices[:25]:
        feat = features[i]
        z = z_scores[i]
        val = real_centroid[i]
        avg = global_mean[i]
        
        direction = "HIGHER" if z > 0 else "LOWER "
        
        # Print the stat: Feature name, actual value, global average, and standard deviation shift
        print(f"  - {feat:<35} | Value: {val:>7.2f} | Avg: {avg:>7.2f} | {direction} by {abs(z):.2f} std devs")
        
print("-" * 60)


--- Cluster Profiles ---

CLUSTER 0 (1 students):
Top 5 defining characteristics vs. the average student:
  - honLST                              | Value:    0.00 | Avg:    0.93 | LOWER  by 3.54 std devs
  - numP                                | Value:    0.00 | Avg:    2.76 | LOWER  by 2.54 std devs
  - averATND                            | Value:    0.00 | Avg:    2.60 | LOWER  by 2.44 std devs
  - numE                                | Value:    0.00 | Avg:    2.72 | LOWER  by 2.31 std devs
  - honHW                               | Value:    0.00 | Avg:    0.78 | LOWER  by 1.87 std devs
  - numQ                                | Value:    0.00 | Avg:    2.31 | LOWER  by 1.86 std devs
  - anonINDX                            | Value:    1.00 | Avg:   28.37 | LOWER  by 1.73 std devs
  - numHW                               | Value:    0.00 | Avg:    1.81 | LOWER  by 1.67 std devs
  - hardHW                              | Value:    0.00 | Avg:    0.38 | LOWER  by 1.66 std devs
  - proxi.2

In [8]:
import numpy as np
import time

print("\n--- Starting Online Learning Phase ---")

# 1. Extract the remaining data starting from index 50000 to the end
online_data = clustering_df.values[50000:]
total_online_obs = len(online_data)

# 2. Process each new observation one by one
for i, row in enumerate(online_data):
    # Record the number of clusters before processing
    num_clusters_before = len(art.W)
    
    # Process the input: classifies, learns, and potentially creates new clusters
    assigned_cluster = art.process_input(row)
    
    # Check if the F2 layer expanded
    num_clusters_after = len(art.W)
    is_new_cluster = (num_clusters_after > num_clusters_before)
    
    # Track the absolute observation number (50001, 50002, etc.)
    obs_num = 50000 + i + 1
    
    # 3. Print what is happening in the network
    if is_new_cluster:
        status_msg = "🆕 NEW CLUSTER CREATED!"
    else:
        status_msg = "🔄 Updated existing.  "
        
    print(f"Obs {obs_num:>6} | Classified to Cluster {assigned_cluster:>3} | {status_msg} | Total F2 Clusters: {num_clusters_after}")
    
    # 4. PAUSE for exactly 1 second so you can watch it learn in real-time
    time.sleep(1)

print("-" * 60)
print(f"--- Online Learning Complete! ---")
print(f"Final F2 Cluster Count: {len(art.W)}")


--- Starting Online Learning Phase ---
------------------------------------------------------------
--- Online Learning Complete! ---
Final F2 Cluster Count: 7


In [9]:
import numpy as np

# 1. Define the network parameters
M = clustering_df.shape[1]      # M = 17 features
limit = 1.0 / np.sqrt(M)        # Upper bound constraint (~0.2425)

# Setting parameters based on the paper's intermediate-learn examples
theta = limit                   # Signal threshold
alpha = limit                   # Uncommitted node choice parameter
rho_star = 0.95                 # Vigilance parameter
beta = 0.01                     # Learning rate (0.01 = slow/intermediate learning)

# 2. Instantiate the model
art = ART2A(M=M, theta=theta, alpha=alpha, rho_star=rho_star, beta=beta)

# 3. Extract the first 50,000 observations
# .copy() ensures we are shuffling our working array without messing up the original DataFrame
data_subset = clustering_df.values[:50000].copy() 

# Set the number of complete passes over the dataset
epochs = 50 

# 4. Run the data through the network with shuffling
print(f"Starting ART 2-A clustering on {len(data_subset)} observations for {epochs} epochs...")
print(f"Mode: Slow Learning (beta = {beta}) with random presentation order")

for epoch in range(epochs):
    print(f"\n--- Epoch {epoch + 1}/{epochs} ---")
    
    # Shuffle the observations before each presentation pass to prevent order bias
    np.random.shuffle(data_subset)
    
    for i, row in enumerate(data_subset):
        art.process_input(row)
        
        # Optional: Print progress every 10,000 rows
        if (i + 1) % 10000 == 0:
            print(f"  Processed {i + 1} rows... Current committed nodes: {len(art.W)}")

# 5. Print the final number of clusters (or nodes)
num_clusters = len(art.W)
print(f"\nTotal number of committed nodes after {epochs} epochs: {num_clusters}")

Starting ART 2-A clustering on 54 observations for 50 epochs...
Mode: Slow Learning (beta = 0.01) with random presentation order

--- Epoch 1/50 ---

--- Epoch 2/50 ---

--- Epoch 3/50 ---

--- Epoch 4/50 ---

--- Epoch 5/50 ---

--- Epoch 6/50 ---

--- Epoch 7/50 ---

--- Epoch 8/50 ---

--- Epoch 9/50 ---

--- Epoch 10/50 ---

--- Epoch 11/50 ---

--- Epoch 12/50 ---

--- Epoch 13/50 ---

--- Epoch 14/50 ---

--- Epoch 15/50 ---

--- Epoch 16/50 ---

--- Epoch 17/50 ---

--- Epoch 18/50 ---

--- Epoch 19/50 ---

--- Epoch 20/50 ---

--- Epoch 21/50 ---

--- Epoch 22/50 ---

--- Epoch 23/50 ---

--- Epoch 24/50 ---

--- Epoch 25/50 ---

--- Epoch 26/50 ---

--- Epoch 27/50 ---

--- Epoch 28/50 ---

--- Epoch 29/50 ---

--- Epoch 30/50 ---

--- Epoch 31/50 ---

--- Epoch 32/50 ---

--- Epoch 33/50 ---

--- Epoch 34/50 ---

--- Epoch 35/50 ---

--- Epoch 36/50 ---

--- Epoch 37/50 ---

--- Epoch 38/50 ---

--- Epoch 39/50 ---

--- Epoch 40/50 ---

--- Epoch 41/50 ---

--- Epoch 42/50 --